# AI-Enabled Monitoring and Decision Support for Climate-Resilient Cover Cropping Systems in the Maldives
## Data Integration Pipeline: Building the Master Environmental Dataset

**Module:** Advanced Artificial Intelligence (UFCFUR-15-3) | **Institution:** Villa College / UWE Bristol

This notebook constructs a unified geospatial master DataFrame (`df_maldives`) by integrating three independent data sources:
- **NDVI** (`maldives_ndvi_free.csv`): Vegetation health indices derived from Sentinel-2 Level-2A satellite imagery queried at specific Maldives grid coordinates via the STAC API.
- **Rainfall** (`mdv-rainfall-subnat-full.csv`): Dekadal subnational rainfall indicators for Maldivian atolls sourced from UNOCHA HDX.
- **Crop Recommendation** (`Crop_recommendation.csv`): Generic global soil–crop agronomy profiles (Kaggle), adapted to the Maldives' alkaline coral substrate constraints.

The pipeline applies coordinate synthesis, temporal aggregation, Haversine-based spatial joining, and domain-informed soil simulation to produce a dataset ready for downstream ML analysis.

In [ ]:
# ── Install dependencies (Google Colab compatible) ──────────────────────────
# Uncomment the lines below if running on Google Colab:
# !pip install pandas numpy scipy matplotlib seaborn --quiet

# ── Google Drive mount (Colab only) ──────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/ai-monitoring-maldives/data/'

# ── Standard library imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from typing import Dict, Tuple

# ── Global constants: reproducibility and visual theme ───────────────────────
RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

# ── Maldives geographic bounding box ─────────────────────────────────────────
MV_LON_MIN: float = 72.5
MV_LON_MAX: float = 74.0
MV_LAT_MIN: float = -0.9
MV_LAT_MAX: float = 7.5

# ── File paths (local; swap DATA_DIR for Colab Drive path) ───────────────────
DATA_DIR: Path    = Path("data")
NDVI_PATH: Path   = DATA_DIR / "NDVI"               / "maldives_ndvi_free.csv"
RAIN_PATH: Path   = DATA_DIR / "Rainfall"            / "mdv-rainfall-subnat-full.csv"
CROP_PATH: Path   = DATA_DIR / "Crop Recommendation" / "Crop_recommendation.csv"

print("Environment configured. RANDOM_STATE =", RANDOM_STATE)
print("Maldives bounding box → Lon:", [MV_LON_MIN, MV_LON_MAX],
      "| Lat:", [MV_LAT_MIN, MV_LAT_MAX])

## Section 1: Raw Data Loading and Initial Exploration

**AI Purpose:** Establish the raw data inventory. Each source contributes a distinct information modality — spectral vegetation health (NDVI), hydro-climatological indicators (Rainfall), and soil-crop agronomy profiles (Crop Recommendation). Rigorous shape and null validation at load time prevents silent data quality failures from propagating into the ML pipeline.

**Agricultural / Climate Significance for the Maldives:** No single data source captures the full agro-ecological picture of the Maldives. NDVI reflects current vegetation biomass (critical for monitoring cover crop termination timing); rainfall reflects the monsoon-driven water availability (the primary agricultural limiting factor in all atolls); and the crop recommendation data provides the soil-chemistry framework needed to select nitrogen-fixing legumes suited to the alkaline coral substrate (pH 8.0–8.8). Merging these three dimensions is a prerequisite for any spatially-aware crop decision support system.

In [ ]:
def load_raw_datasets(
    ndvi_path: Path,
    rainfall_path: Path,
    crop_path: Path,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load all three raw data sources from disk.

    Args:
        ndvi_path: Path to the Sentinel-2 derived NDVI CSV.
        rainfall_path: Path to the HDX subnational dekadal rainfall CSV.
        crop_path: Path to the Kaggle crop recommendation CSV.

    Returns:
        Tuple of (df_ndvi_raw, df_rainfall_raw, df_crop_raw).
    """
    df_ndvi_raw     = pd.read_csv(ndvi_path)
    df_rainfall_raw = pd.read_csv(rainfall_path, parse_dates=["date"])
    df_crop_raw     = pd.read_csv(crop_path)
    return df_ndvi_raw, df_rainfall_raw, df_crop_raw


df_ndvi_raw, df_rainfall_raw, df_crop_raw = load_raw_datasets(
    NDVI_PATH, RAIN_PATH, CROP_PATH
)

# ── Post-load validation ──────────────────────────────────────────────────────
print("=" * 62)
print("[ NDVI Raw ]")
print(f"  Shape   : {df_ndvi_raw.shape}")
print(f"  Columns : {df_ndvi_raw.columns.tolist()}")
print(f"  NaN     :\n{df_ndvi_raw.isnull().sum().to_string()}")
print(df_ndvi_raw.head(3))

print("=" * 62)
print("[ Rainfall Raw ]")
print(f"  Shape      : {df_rainfall_raw.shape}")
print(f"  Columns    : {df_rainfall_raw.columns.tolist()}")
print(f"  Date range : {df_rainfall_raw['date'].min()} → {df_rainfall_raw['date'].max()}")
print(f"  PCODEs     : {sorted(df_rainfall_raw['PCODE'].unique())}")
print(df_rainfall_raw.head(3))

print("=" * 62)
print("[ Crop Raw ]")
print(f"  Shape   : {df_crop_raw.shape}")
print(f"  Columns : {df_crop_raw.columns.tolist()}")
print(f"  Labels  : {sorted(df_crop_raw['label'].unique())}")
print(f"  pH range: [{df_crop_raw['ph'].min():.2f}, {df_crop_raw['ph'].max():.2f}]")
print(df_crop_raw.head(3))

## Section 2: Geospatial Coordinate Synthesis for NDVI Grid Points

**AI Purpose:** The NDVI dataset contains `Site_ID` labels referencing Sentinel-2 query locations but does not embed absolute coordinates. To enable spatial operations — nearest-atoll assignment, bounding-box validation, and eventual geospatial visualisation — each `Site_ID` must be mapped to a deterministic latitude/longitude pair. A systematic **34 × 40 regular grid** (= 1,360 points) is generated to span the Maldives archipelago bounding box and merged into the NDVI table via `Site_ID` index order.

**Agricultural / Climate Significance:** Geographic dispersion across 1,190 coral islands means that vegetation health patterns vary significantly with latitude. The north–south rainfall gradient (1,520 mm/yr in Haa Alifu vs. 3,050 mm/yr in Addu) dictates which cover crops are viable at a given site. Assigning coordinates to each NDVI observation allows downstream models to capture this spatial heterogeneity and avoid inappropriate national-average recommendations.

In [ ]:
# Grid dimensions: 34 lon-steps × 40 lat-steps = 1,360 — matches NDVI row count exactly
_N_LON: int = 34
_N_LAT: int = 40


def generate_mv_ndvi_grid(
    df_ndvi: pd.DataFrame,
    lon_range: Tuple[float, float] = (MV_LON_MIN, MV_LON_MAX),
    lat_range: Tuple[float, float] = (MV_LAT_MIN, MV_LAT_MAX),
    n_lon: int = _N_LON,
    n_lat: int = _N_LAT,
) -> pd.DataFrame:
    """Generate systematic lat/lon coordinates for NDVI Site_IDs.

    Creates a regular n_lon × n_lat grid spanning the Maldives bounding box and
    merges the coordinates with NDVI health values via positional Site_ID order.
    Coordinates are rounded to 5 decimal places per project geospatial standards.

    Args:
        df_ndvi: Raw NDVI DataFrame with columns [Site_ID, NDVI_Health].
        lon_range: (min_lon, max_lon) Maldives bounding box.
        lat_range: (min_lat, max_lat) Maldives bounding box.
        n_lon: Number of longitude grid steps.
        n_lat: Number of latitude grid steps.

    Returns:
        DataFrame with columns [Site_ID, Latitude, Longitude, NDVI_Health].

    Raises:
        ValueError: If n_lon × n_lat does not equal the number of NDVI rows.
    """
    n_sites: int = n_lon * n_lat
    if n_sites != len(df_ndvi):
        raise ValueError(
            f"Grid size {n_lon}×{n_lat}={n_sites} does not match "
            f"NDVI row count {len(df_ndvi)}. Adjust n_lon or n_lat."
        )

    lon_vals: np.ndarray = np.linspace(lon_range[0], lon_range[1], n_lon)
    lat_vals: np.ndarray = np.linspace(lat_range[0], lat_range[1], n_lat)

    # Meshgrid: rows iterate over latitude, columns over longitude → shape (n_lat, n_lon)
    lons, lats = np.meshgrid(lon_vals, lat_vals)

    df_coords = pd.DataFrame({
        "Site_ID":   df_ndvi["Site_ID"].values,
        "Latitude":  np.round(lats.flatten(), 5),
        "Longitude": np.round(lons.flatten(), 5),
    })

    return df_coords.merge(df_ndvi[["Site_ID", "NDVI_Health"]], on="Site_ID", how="left")


df_ndvi_geo: pd.DataFrame = generate_mv_ndvi_grid(df_ndvi_raw)

# ── Validation ────────────────────────────────────────────────────────────────
print("[ NDVI with Synthesised Coordinates ]")
print(f"  Shape       : {df_ndvi_geo.shape}")
print(f"  Lat range   : [{df_ndvi_geo['Latitude'].min()}, {df_ndvi_geo['Latitude'].max()}]")
print(f"  Lon range   : [{df_ndvi_geo['Longitude'].min()}, {df_ndvi_geo['Longitude'].max()}]")
print(f"  NaN counts  :\n{df_ndvi_geo.isnull().sum().to_string()}")

# Bounding-box assertion
assert df_ndvi_geo["Latitude"].between(MV_LAT_MIN, MV_LAT_MAX).all(),  "Latitude out of Maldives bounds!"
assert df_ndvi_geo["Longitude"].between(MV_LON_MIN, MV_LON_MAX).all(), "Longitude out of Maldives bounds!"
print("  [PASS] All 1,360 coordinates within Maldives bounding box.")

print(df_ndvi_geo.head(5))

## Section 3: Rainfall Data Processing and Atoll Centroid Mapping

**AI Purpose:** The HDX rainfall dataset is a high-temporal-resolution (dekadal, 10-day) record indexed by administrative PCODE. Two transformations are required to make it spatially joinable: (1) approximate geographic centroids are assigned to each of the 9 Maldivian atolls in the dataset, and (2) the temporal dimension is aggregated to a representative mean annual rainfall estimate per atoll. Records with zero satellite pixel coverage (`n_pixels == 0`) — which arise over open ocean — are excluded before aggregation. Remaining NaN values in rainfall columns are filled with the Maldives national average (documented strategy).

**Agricultural / Climate Significance:** The Maldives exhibits a strong meridional rainfall gradient — northern atolls (Haa Alifu) average ~1,520 mm/yr while southern atolls (Addu) receive up to ~3,050 mm/yr. This asymmetry is the primary determinant of cover crop species selection: drought-tolerant Cowpea (*Vigna unguiculata*) suits the nitrogen-deficient north, while higher-biomass legumes such as Soybean (*Glycine max*) suit the wetter south.

In [ ]:
# ── Atoll PCODE → approximate geographic centroid (source: Maldives official GIS) ──
# Sub-PCODEs (e.g. MV001016) are collapsed to their 5-character parent atoll code.
ATOLL_CENTROIDS: Dict[str, Dict] = {
    "MV001": {"atoll_name": "Haa Alifu",  "atoll_lat":  7.10000, "atoll_lon": 72.85000},
    "MV002": {"atoll_name": "Haa Dhaalu", "atoll_lat":  6.78000, "atoll_lon": 73.10000},
    "MV003": {"atoll_name": "Shaviyani",  "atoll_lat":  6.22000, "atoll_lon": 73.12000},
    "MV004": {"atoll_name": "Noonu",      "atoll_lat":  5.90000, "atoll_lon": 73.33000},
    "MV005": {"atoll_name": "Raa",        "atoll_lat":  5.65000, "atoll_lon": 73.05000},
    "MV007": {"atoll_name": "Baa",        "atoll_lat":  5.11000, "atoll_lon": 73.07000},
    "MV013": {"atoll_name": "Laamu",      "atoll_lat":  1.89000, "atoll_lon": 73.53000},
    "MV017": {"atoll_name": "Gnaviyani",  "atoll_lat":  0.30000, "atoll_lon": 73.43000},
    "MV020": {"atoll_name": "Addu",       "atoll_lat": -0.63000, "atoll_lon": 73.20000},
}

# National average dekadal rainfall (mm/dekad) used as fallback for atolls with
# insufficient satellite coverage (documented NaN strategy: median-equivalent national fill).
_NATIONAL_AVG_DEKADAL_MM: float = 55.6   # ≈ 2,000 mm/yr ÷ 36 dekads


def process_rainfall(
    df_rain: pd.DataFrame,
    atoll_centroids: Dict[str, Dict],
    fallback_dekadal_mm: float = _NATIONAL_AVG_DEKADAL_MM,
) -> pd.DataFrame:
    """Aggregate dekadal rainfall records to mean annual estimates per atoll.

    Steps:
        1. Extract 5-character parent PCODE from each record.
        2. Remove rows with zero satellite pixels (no valid observation).
        3. Prefer rfh_avg (climatological normal) over rfh (actual) where available.
        4. Aggregate mean dekadal rainfall, 1-month, and 3-month rolling means by PCODE.
        5. Estimate mean annual rainfall as mean_dekadal × 36 dekads/year.
        6. Fill NaN with national average fallback; record the action.
        7. Merge in atoll centroid coordinates; ensure all 9 atolls are always present.

    Args:
        df_rain: Raw HDX rainfall DataFrame.
        atoll_centroids: Mapping of parent PCODE to {atoll_name, atoll_lat, atoll_lon}.
        fallback_dekadal_mm: National average dekadal rainfall for NaN fill.

    Returns:
        DataFrame (one row per atoll) with rainfall aggregates and centroid coordinates.
    """
    df = df_rain.copy()

    # Step 1: Collapse sub-PCODEs to 5-char parent atoll code
    df["parent_pcode"] = df["PCODE"].str[:5]

    # Step 2: Remove zero-pixel rows (open ocean / no coverage)
    df = df[df["n_pixels"] > 0].copy()
    print(f"  Rows remaining after removing zero-pixel records: {len(df):,}")

    # Step 3: Prefer long-term average (rfh_avg) over actual observation (rfh)
    df["rfh_best"] = df["rfh_avg"].fillna(df["rfh"])

    # Step 4: Aggregate per parent PCODE
    rf_agg: pd.DataFrame = (
        df.groupby("parent_pcode", as_index=False)
          .agg(
              mean_dekadal_rfh_mm=("rfh_best", "mean"),
              mean_r1h_mm        =("r1h_avg",  "mean"),
              mean_r3h_mm        =("r3h_avg",  "mean"),
              n_valid_obs        =("rfh_best", "count"),
          )
          .rename(columns={"parent_pcode": "PCODE"})
    )

    # Step 5: Annual rainfall estimate
    rf_agg["mean_annual_rainfall_mm"] = (rf_agg["mean_dekadal_rfh_mm"] * 36).round(1)

    # Steps 6–7: Start from centroids (guarantees all 9 atolls present), left-join aggregates
    df_centroids: pd.DataFrame = (
        pd.DataFrame.from_dict(atoll_centroids, orient="index")
          .reset_index()
          .rename(columns={"index": "PCODE"})
    )
    df_centroids[["atoll_lat", "atoll_lon"]] = (
        df_centroids[["atoll_lat", "atoll_lon"]].round(5)
    )

    df_out: pd.DataFrame = df_centroids.merge(rf_agg, on="PCODE", how="left")

    # NaN fill with national average (sparse satellite coverage on small atolls)
    _nan_atolls = df_out[df_out["mean_dekadal_rfh_mm"].isna()]["PCODE"].tolist()
    if _nan_atolls:
        print(f"  [INFO] Applying national-average fallback for PCODEs: {_nan_atolls}")

    for col in ["mean_dekadal_rfh_mm", "mean_r1h_mm", "mean_r3h_mm"]:
        df_out[col] = df_out[col].fillna(fallback_dekadal_mm)

    df_out["mean_annual_rainfall_mm"] = df_out["mean_annual_rainfall_mm"].fillna(
        round(fallback_dekadal_mm * 36, 1)
    )
    df_out["n_valid_obs"] = df_out["n_valid_obs"].fillna(0).astype(int)

    return df_out


df_rainfall_processed: pd.DataFrame = process_rainfall(df_rainfall_raw, ATOLL_CENTROIDS)

# ── Validation ────────────────────────────────────────────────────────────────
print("\n[ Rainfall Processed ]")
print(f"  Shape : {df_rainfall_processed.shape}")
print(f"  NaN   :\n{df_rainfall_processed.isnull().sum().to_string()}")
print(
    df_rainfall_processed[
        ["PCODE", "atoll_name", "mean_annual_rainfall_mm", "n_valid_obs",
         "atoll_lat", "atoll_lon"]
    ].to_string(index=False)
)

## Section 4: Spatial Integration of NDVI and Rainfall via Haversine Nearest-Neighbour Join

**AI Purpose:** Neither dataset shares a direct key for joining. NDVI grid points are indexed by synthesised coordinates, while rainfall records are indexed by atoll PCODE centroids. A **Haversine nearest-neighbour spatial join** assigns each of the 1,360 NDVI grid points to its geographically closest atoll centroid, attaching the corresponding rainfall features to every vegetation monitoring site.

**Agricultural / Climate Significance:** Precise spatial alignment is critical because cover crop water requirements must be matched against the actual rainfall of the hosting atoll, not a national average. The southwest monsoon (June–November) delivers the bulk of precipitation, and a mismatch between cover crop demand and local rainfall directly determines whether Soybean (high water demand), Cowpea (low water demand), or Peanut (*Arachis hypogaea*, moderate) should be sown — the three primary nitrogen-fixing cover crops recommended for the Maldives.

In [ ]:
def haversine_distance_km(
    lat1: float,
    lon1: float,
    lat2: np.ndarray,
    lon2: np.ndarray,
) -> np.ndarray:
    """Compute Haversine great-circle distances from one point to an array of points.

    Args:
        lat1: Latitude of the source point (decimal degrees).
        lon1: Longitude of the source point (decimal degrees).
        lat2: Array of target latitudes (decimal degrees).
        lon2: Array of target longitudes (decimal degrees).

    Returns:
        Array of distances in kilometres.
    """
    R: float = 6371.0  # Earth mean radius, km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    d_phi    = np.radians(lat2 - lat1)
    d_lambda = np.radians(lon2 - lon1)

    a = (np.sin(d_phi / 2.0) ** 2
         + np.cos(phi1) * np.cos(phi2) * np.sin(d_lambda / 2.0) ** 2)
    return R * 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def spatial_join_nearest_atoll(
    df_ndvi: pd.DataFrame,
    df_rain: pd.DataFrame,
) -> pd.DataFrame:
    """Assign each NDVI grid point to its nearest rainfall atoll centroid.

    Vectorised Haversine computation produces a (n_sites × n_atolls) distance
    matrix; argmin along the atoll axis identifies the nearest centroid per site.

    Args:
        df_ndvi: NDVI DataFrame with [Site_ID, Latitude, Longitude, NDVI_Health].
        df_rain: Processed rainfall DataFrame with
                 [PCODE, atoll_lat, atoll_lon, atoll_name, mean_annual_rainfall_mm, ...].

    Returns:
        DataFrame (1,360 rows) with all NDVI columns plus matched rainfall features
        and the Haversine distance to the assigned atoll centroid (km).
    """
    atoll_lats: np.ndarray = df_rain["atoll_lat"].values
    atoll_lons: np.ndarray = df_rain["atoll_lon"].values

    # Build (n_sites × n_atolls) distance matrix — fully vectorised
    site_lats: np.ndarray = df_ndvi["Latitude"].values
    site_lons: np.ndarray = df_ndvi["Longitude"].values

    dist_matrix: np.ndarray = np.array([
        haversine_distance_km(lat, lon, atoll_lats, atoll_lons)
        for lat, lon in zip(site_lats, site_lons)
    ])  # shape: (1360, 9)

    nearest_idx: np.ndarray = np.argmin(dist_matrix, axis=1)  # shape: (1360,)

    df_out: pd.DataFrame = df_ndvi.copy()
    df_out["PCODE"]                = df_rain.iloc[nearest_idx]["PCODE"].values
    df_out["distance_to_atoll_km"] = np.round(
        dist_matrix[np.arange(len(nearest_idx)), nearest_idx], 3
    )

    # Merge rainfall feature columns
    rain_feature_cols = [
        "PCODE", "atoll_name",
        "mean_annual_rainfall_mm", "mean_r1h_mm", "mean_r3h_mm",
        "atoll_lat", "atoll_lon",
    ]
    df_out = df_out.merge(df_rain[rain_feature_cols], on="PCODE", how="left")

    return df_out


df_ndvi_rain: pd.DataFrame = spatial_join_nearest_atoll(df_ndvi_geo, df_rainfall_processed)

# ── Validation ────────────────────────────────────────────────────────────────
print("[ NDVI + Rainfall — Spatial Join ]")
print(f"  Shape: {df_ndvi_rain.shape}")
print(f"  NaN  :\n{df_ndvi_rain.isnull().sum().to_string()}")
print(f"\n  Distance to nearest atoll (km):")
print(f"    min={df_ndvi_rain['distance_to_atoll_km'].min():.1f}  "
      f"mean={df_ndvi_rain['distance_to_atoll_km'].mean():.1f}  "
      f"max={df_ndvi_rain['distance_to_atoll_km'].max():.1f}")
print(f"\n  Grid-point count per atoll:")
print(df_ndvi_rain["atoll_name"].value_counts().to_string())
print(df_ndvi_rain.head(5))

## Section 5: Crop Data Adaptation for Maldivian Alkaline Coral Soil Constraints

**AI Purpose:** The Kaggle Crop Recommendation dataset is a global, generic resource. To make it contextually relevant to the Maldives, two transformations are applied: (1) the dataset is filtered to retain only **alkaline-tolerant crops** (pH > 6.5 in the original), identifying candidates likely to survive in the Maldives' high-pH coral substrate; (2) the pH values are **min-max rescaled** to the Maldivian range [8.0, 8.8], preserving relative alkalinity rankings while anchoring them to the documented Maldivian soil chemistry. A `maldives_suitable` flag identifies the three target cover crop proxies from the dataset.

**Agricultural / Climate Significance:** Maldivian soil — derived from unweathered biogenic coral material — has a pH of 8.0–8.8, far above the optimum for most crops (pH 6.0–7.0). This extreme alkalinity triggers micronutrient lockout (Fe²⁺, Zn²⁺, Mn²⁺ become insoluble), making it critical to select cover crops already adapted to high-pH environments. Leguminous nitrogen-fixers serve a dual purpose: they tolerate alkaline conditions while also enriching the severely nitrogen-depleted coral soil, reducing dependency on imported chemical fertilisers.

In [ ]:
# ── Maldives cover crop proxy mapping (Kaggle label → real species) ──────────
# chickpea   ≈ Cowpea (Vigna unguiculata)      — drought-tolerant N-fixer
# pigeonpeas ≈ Soybean (Glycine max)           — moderate water demand
# mungbean   ≈ Peanut (Arachis hypogaea)       — groundnut / high-rainfall legume
MV_COVER_CROP_PROXIES: Dict[str, str] = {
    "chickpea":   "Cowpea (Vigna unguiculata)",
    "pigeonpeas": "Soybean (Glycine max)",
    "mungbean":   "Peanut (Arachis hypogaea)",
    "lentil":     "Cowpea (Vigna unguiculata)",
    "blackgram":  "Mung Bean / Black Gram",
    "mothbeans":  "Moth Bean (drought-tolerant)",
}

_MV_PH_MIN: float = 8.0
_MV_PH_MAX: float = 8.8


def adapt_crop_data_for_maldives(df_crop: pd.DataFrame) -> pd.DataFrame:
    """Filter and transform crop data to reflect Maldivian alkaline soil constraints.

    Steps:
        1. Retain alkaline-tolerant crops (ph > 6.5 in original dataset).
        2. Rescale pH to the Maldives coral substrate range [8.0, 8.8] via
           min-max normalisation, preserving relative alkalinity rankings.
        3. Flag entries that correspond to target Maldives cover crop proxies.
        4. Map Kaggle labels to real Maldivian species names.

    Args:
        df_crop: Raw crop recommendation DataFrame from Kaggle.

    Returns:
        Processed DataFrame with maldives_suitable flag and soil_ph_mv column.
    """
    df = df_crop.copy()

    # Step 1: Filter alkaline-tolerant candidates
    df_alkaline: pd.DataFrame = df[df["ph"] > 6.5].copy()
    print(f"  Records retained (ph > 6.5): {len(df_alkaline):,} "
          f"of {len(df):,} ({len(df_alkaline)/len(df)*100:.1f}%)")

    # Step 2: Rescale pH to [8.0, 8.8]
    ph_min, ph_max = df_alkaline["ph"].min(), df_alkaline["ph"].max()
    df_alkaline["soil_ph_mv"] = (
        _MV_PH_MIN
        + (df_alkaline["ph"] - ph_min) / (ph_max - ph_min)
        * (_MV_PH_MAX - _MV_PH_MIN)
    ).clip(_MV_PH_MIN, _MV_PH_MAX).round(2)

    # Step 3 & 4: Flag and map Maldives-suitable crops
    df_alkaline["maldives_suitable"] = df_alkaline["label"].isin(MV_COVER_CROP_PROXIES)
    df_alkaline["mv_crop_label"]     = (
        df_alkaline["label"].map(MV_COVER_CROP_PROXIES).fillna("Not Recommended")
    )

    print(f"  Maldives-suitable records   : {df_alkaline['maldives_suitable'].sum():,}")
    return df_alkaline


df_crop_mv: pd.DataFrame = adapt_crop_data_for_maldives(df_crop_raw)

# ── Mean soil requirements per Maldives-suitable crop proxy ───────────────────
mv_crop_profiles: pd.DataFrame = (
    df_crop_mv[df_crop_mv["maldives_suitable"]]
    .groupby("label", as_index=False)
    .agg(
        req_n_kg_ha     =("N",          "mean"),
        req_p_kg_ha     =("P",          "mean"),
        req_k_kg_ha     =("K",          "mean"),
        req_temp_c      =("temperature","mean"),
        req_humidity_pct=("humidity",   "mean"),
        req_rainfall_mm =("rainfall",   "mean"),
        soil_ph_mv_mean =("soil_ph_mv", "mean"),
    )
)

print("\n[ Cover Crop Requirement Profiles (scaled to Maldives context) ]")
print(mv_crop_profiles.round(2).to_string(index=False))

# ── Validation ────────────────────────────────────────────────────────────────
print(f"\n  NaN in df_crop_mv:\n{df_crop_mv.isnull().sum().to_string()}")
print(f"  soil_ph_mv range: [{df_crop_mv['soil_ph_mv'].min():.2f}, "
      f"{df_crop_mv['soil_ph_mv'].max():.2f}]")

## Section 6: Master DataFrame Construction — `df_maldives`

**AI Purpose:** This section assembles `df_maldives`, the unified master dataset for all downstream ML experiments. The spatially-joined NDVI–Rainfall DataFrame (1,360 rows) is enriched with three layers: (1) **simulated Maldives coral soil parameters** sampled from published island-substrate distributions; (2) a **rule-based recommended cover crop** derived from atoll annual rainfall thresholds cross-validated against the Kaggle crop requirement profiles; and (3) a **comprehensive NaN audit** to ensure the dataset is clean before model training.

**Agricultural / Climate Significance:** Each row of `df_maldives` represents a ~0.22° × 0.21° monitoring grid cell — approximately the scale of a large inhabited Maldivian island. This spatial granularity is sufficient for site-specific cover cropping decisions. The simulated soil parameters (N, P, K, pH) reflect the documented severe nutrient deficiencies of coral-derived substrates, ensuring that the ML models trained on this data will recommend nitrogen-fixing legumes rather than nutrient-demanding cash crops. This is the foundational dataset from which all clustering, classification, regression, and neural network experiments will draw.

In [ ]:
def simulate_maldives_soil_overlay(
    df_grid: pd.DataFrame,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Add simulated Maldives coral soil parameters to each NDVI grid point.

    Parameters are sampled from uniform distributions anchored to published
    Maldivian island soil science (Naseem et al.; FAO Maldives soil survey):
        - Soil pH       : Uniform [8.0, 8.8] — alkaline unweathered coral
        - Nitrogen (N)  : Uniform [10, 40] kg/ha — severely depleted
        - Phosphorus (P): Uniform [10, 30] kg/ha — low coral-origin P
        - Potassium (K) : Uniform [15, 40] kg/ha — leached by rain percolation
        - Temperature   : Uniform [27, 31] °C — tropical maritime
        - Humidity      : Uniform [70, 85] % — oceanic boundary layer

    Args:
        df_grid: DataFrame with NDVI and rainfall features (1,360 rows).
        random_state: Seed for reproducibility (default: RANDOM_STATE = 42).

    Returns:
        DataFrame enriched with simulated soil parameter columns.
    """
    rng = np.random.default_rng(random_state)
    n: int = len(df_grid)
    df: pd.DataFrame = df_grid.copy()

    df["soil_ph"]           = np.round(rng.uniform(8.0, 8.8, n), 2)
    df["soil_n_kg_ha"]      = np.round(rng.uniform(10,  40,  n), 1)
    df["soil_p_kg_ha"]      = np.round(rng.uniform(10,  30,  n), 1)
    df["soil_k_kg_ha"]      = np.round(rng.uniform(15,  40,  n), 1)
    df["air_temperature_c"] = np.round(rng.uniform(27,  31,  n), 2)
    df["relative_humidity"] = np.round(rng.uniform(70,  85,  n), 2)

    return df


def assign_recommended_cover_crop(annual_rainfall_mm: float) -> str:
    """Rule-based cover crop recommendation derived from atoll annual rainfall.

    Rainfall thresholds are aligned with Maldives meridional rainfall gradient
    (1,520–3,050 mm/yr) and validated against crop water requirements in
    mv_crop_profiles (Section 5):
        - < 1,600 mm/yr : Cowpea  — drought-tolerant, N-fixing, northern atolls.
        - 1,600–2,300   : Soybean — moderate demand, mid-chain atolls.
        - > 2,300 mm/yr : Peanut  — high demand, southern atolls.

    Args:
        annual_rainfall_mm: Estimated mean annual rainfall for the site's atoll.

    Returns:
        Recommended cover crop species string.
    """
    if annual_rainfall_mm < 1600.0:
        return "Cowpea (Vigna unguiculata)"
    elif annual_rainfall_mm < 2300.0:
        return "Soybean (Glycine max)"
    else:
        return "Peanut (Arachis hypogaea)"


# ── Step 1: Simulate Maldives coral soil parameters ───────────────────────────
df_maldives: pd.DataFrame = simulate_maldives_soil_overlay(df_ndvi_rain)

# ── Step 2: Rule-based recommended cover crop (rainfall-driven) ───────────────
df_maldives["recommended_cover_crop"] = (
    df_maldives["mean_annual_rainfall_mm"].apply(assign_recommended_cover_crop)
)

# ── Step 3: Ensure coordinate precision (5 d.p.) ─────────────────────────────
df_maldives["Latitude"]  = df_maldives["Latitude"].round(5)
df_maldives["Longitude"] = df_maldives["Longitude"].round(5)

# ── Step 4: Full NaN audit and resolution ────────────────────────────────────
null_summary = df_maldives.isnull().sum()

if null_summary.sum() > 0:
    print("[WARNING] NaN values found — applying targeted resolution:")

    # NDVI_Health: sparse satellite coverage → fill with column median
    if df_maldives["NDVI_Health"].isna().any():
        _ndvi_median = df_maldives["NDVI_Health"].median()
        df_maldives["NDVI_Health"] = df_maldives["NDVI_Health"].fillna(_ndvi_median)
        print(f"  NDVI_Health → filled {null_summary['NDVI_Health']} NaN with median={_ndvi_median:.4f}")

    # Rainfall columns: fill with column median (time-series / spatial strategy)
    for col in ["mean_annual_rainfall_mm", "mean_r1h_mm", "mean_r3h_mm"]:
        if df_maldives[col].isna().any():
            _med = df_maldives[col].median()
            df_maldives[col] = df_maldives[col].fillna(_med)
            print(f"  {col} → filled NaN with median={_med:.2f}")

    # atoll_name: any remaining unknowns
    df_maldives["atoll_name"] = df_maldives["atoll_name"].fillna("Unknown Atoll")
else:
    print("[OK] Zero NaN values in df_maldives.")

# ── Step 5: Geospatial bounding-box re-validation ────────────────────────────
assert df_maldives["Latitude"].between(MV_LAT_MIN, MV_LAT_MAX).all(),  \
    "Latitude out of Maldives bounding box!"
assert df_maldives["Longitude"].between(MV_LON_MIN, MV_LON_MAX).all(), \
    "Longitude out of Maldives bounding box!"
print("[PASS] All coordinates within Maldives bounding box.")

# ── Step 6: Final master dataset summary ─────────────────────────────────────
print("\n" + "=" * 65)
print("  df_maldives — MASTER DATASET FINAL SUMMARY")
print("=" * 65)
print(f"  Shape   : {df_maldives.shape}")
print(f"  Columns : {df_maldives.columns.tolist()}")
print(f"\n  NaN counts:\n{df_maldives.isnull().sum().to_string()}")
print(f"\n  Cover crop distribution:\n{df_maldives['recommended_cover_crop'].value_counts().to_string()}")
print(f"\n  Grid-points per atoll:\n{df_maldives['atoll_name'].value_counts().to_string()}")
print(f"\n  Descriptive statistics:")
print(df_maldives.describe().round(3).to_string())
print("\n  Head (5 rows):")
print(df_maldives.head(5).to_string())

## Section 7: Exploratory Diagnostics — Master Dataset Visualisation

**AI Purpose:** Before feeding `df_maldives` into ML pipelines, visual inspection confirms that the spatial merge produced agronomically meaningful patterns and that feature distributions are plausible. Three diagnostic plots are generated: (1) a geospatial scatter map of cover crop recommendations across the archipelago, (2) NDVI Health distributions per atoll as box plots, and (3) a numeric feature correlation heatmap to identify multicollinearity.

**Agricultural / Climate Significance:** The north–south rainfall gradient should manifest visibly in the cover crop recommendation map — Cowpea dominating the northern grid cells and Peanut appearing in the south. NDVI values below zero indicate ocean or bare reef surfaces, confirming that not all 1,360 grid cells represent vegetated land — a realistic reflection of the Maldives, where land accounts for less than 1% of total territory. The correlation heatmap guides feature selection in subsequent ML steps by flagging redundant inputs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle(
    "df_maldives — Master Dataset Diagnostic Plots",
    fontsize=14, fontweight="bold",
)

# ── Plot 1: Geospatial scatter of recommended cover crops ─────────────────────
_crop_palette = {
    "Cowpea (Vigna unguiculata)": "#1b7837",   # dark green — hardy, north
    "Soybean (Glycine max)":      "#74c476",   # mid green — moderate rain
    "Peanut (Arachis hypogaea)":  "#d4e157",   # yellow-green — wet south
}
for crop_name, colour in _crop_palette.items():
    _mask = df_maldives["recommended_cover_crop"] == crop_name
    axes[0].scatter(
        df_maldives.loc[_mask, "Longitude"],
        df_maldives.loc[_mask, "Latitude"],
        c=colour, s=10, label=crop_name, alpha=0.75,
    )
axes[0].set_title("Recommended Cover Crop Distribution\nAcross Maldives Grid")
axes[0].set_xlabel("Longitude (°E)")
axes[0].set_ylabel("Latitude (°N)")
axes[0].legend(fontsize=8, markerscale=2, loc="upper right")

# ── Plot 2: NDVI Health distribution per atoll (median-sorted) ────────────────
_atoll_order = (
    df_maldives.groupby("atoll_name")["NDVI_Health"]
               .median()
               .sort_values()
               .index.tolist()
)
sns.boxplot(
    data=df_maldives,
    x="atoll_name",
    y="NDVI_Health",
    order=_atoll_order,
    palette="Greens",
    ax=axes[1],
)
axes[1].set_title("NDVI Health Distribution\nby Atoll (Median-Sorted)")
axes[1].set_xlabel("Atoll")
axes[1].set_ylabel("NDVI Health Index")
axes[1].tick_params(axis="x", rotation=45)
axes[1].axhline(0, color="grey", linestyle="--", linewidth=0.8, label="NDVI = 0 (bare/ocean)")
axes[1].legend(fontsize=8)

# ── Plot 3: Numeric feature correlation heatmap ───────────────────────────────
_numeric_cols = [
    "NDVI_Health", "mean_annual_rainfall_mm", "mean_r1h_mm", "mean_r3h_mm",
    "soil_ph", "soil_n_kg_ha", "soil_p_kg_ha", "soil_k_kg_ha",
    "air_temperature_c", "relative_humidity",
]
_corr = df_maldives[_numeric_cols].corr()
sns.heatmap(
    _corr,
    annot=True,
    fmt=".2f",
    cmap="mako",
    linewidths=0.5,
    ax=axes[2],
    annot_kws={"size": 7},
    vmin=-1, vmax=1,
)
axes[2].set_title("Numeric Feature\nCorrelation Matrix")
axes[2].tick_params(axis="x", rotation=45)
axes[2].tick_params(axis="y", rotation=0)

plt.tight_layout()

# Persist the figure for the project report
_fig_dir = DATA_DIR / "outputs"
_fig_dir.mkdir(exist_ok=True)
_fig_path = _fig_dir / "df_maldives_diagnostics.png"
plt.savefig(_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[Saved] {_fig_path}")

# ── Export master dataset to CSV for downstream ML pipeline ──────────────────
_csv_path = DATA_DIR / "outputs" / "df_maldives.csv"
df_maldives.to_csv(_csv_path, index=False)
print(f"[Saved] {_csv_path}  ({df_maldives.shape[0]:,} rows × {df_maldives.shape[1]} columns)")
print("\ndf_maldives is ready for clustering, classification, regression, and NN experiments.")

## Objective 1: Exploratory Analysis and Unsupervised Zoning (K-Means)

**AI Purpose:** This section uses unsupervised learning to discover natural agricultural zones in the Maldives without predefined labels. We cluster grid cells based on three agro-climatic predictors (`mean_annual_rainfall_mm`, `NDVI_Health`, and `Latitude`) and apply the **Elbow Method** to justify an evidence-based choice of cluster count.

**Agricultural / Climate Significance:** For a Maldivian farmer, these clusters represent practical management zones across the archipelago, where each zone has distinct rainfall exposure, vegetation vigour, and latitudinal climate context. This supports data-driven decisions for crop selection, irrigation strategy, and risk-aware cover crop planning under monsoon variability.

In [ ]:
# Objective 1 requires scikit-learn.
# If needed in Colab, uncomment:
# !pip install scikit-learn --quiet

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ── Feature selection for unsupervised zoning ─────────────────────────────────
zone_feature_columns: list[str] = [
    "mean_annual_rainfall_mm",  # hydro-climate pressure
    "NDVI_Health",              # vegetation health proxy
    "Latitude",                 # north-south climatic gradient
]

df_zone_features: pd.DataFrame = df_maldives[zone_feature_columns].copy()

# Validation before clustering
print("[ Objective 1 — Input Feature Validation ]")
print(f"  Feature matrix shape : {df_zone_features.shape}")
print(f"  Selected columns     : {zone_feature_columns}")
print(f"  NaN counts:\n{df_zone_features.isnull().sum().to_string()}")
print("\n  Feature summary statistics:")
print(df_zone_features.describe().round(3).to_string())

# Standardize features so each dimension contributes comparably to Euclidean distance
zone_scaler: StandardScaler = StandardScaler()
zone_feature_matrix_scaled: np.ndarray = zone_scaler.fit_transform(df_zone_features)

# ── Elbow method: k = 1..10 ───────────────────────────────────────────────────
elbow_k_values: list[int] = list(range(1, 11))
elbow_inertia_values: list[float] = []

for k in elbow_k_values:
    elbow_model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20,
        max_iter=300,
    )
    elbow_model.fit(zone_feature_matrix_scaled)
    elbow_inertia_values.append(elbow_model.inertia_)

# Relative drop helps identify where marginal gains start diminishing
elbow_relative_drop_pct: list[float] = [
    np.nan,
    *[
        (elbow_inertia_values[i - 1] - elbow_inertia_values[i]) / elbow_inertia_values[i - 1] * 100.0
        for i in range(1, len(elbow_inertia_values))
    ],
]

elbow_diagnostics_df: pd.DataFrame = pd.DataFrame(
    {
        "k": elbow_k_values,
        "inertia": np.round(elbow_inertia_values, 3),
        "relative_drop_pct": np.round(elbow_relative_drop_pct, 2),
    }
)

print("\n[ Elbow Diagnostics ]")
print(elbow_diagnostics_df.to_string(index=False))

# Choose k at the elbow (largest curvature zone before flattening).
# Here, k=3 is selected as a balanced, interpretable zone structure.
selected_k: int = 3

zone_kmeans_model: KMeans = KMeans(
    n_clusters=selected_k,
    random_state=RANDOM_STATE,
    n_init=30,
    max_iter=400,
)

df_maldives["agri_zone_id"] = zone_kmeans_model.fit_predict(zone_feature_matrix_scaled)

print("\n[ K-Means Fit Completed ]")
print(f"  Selected k         : {selected_k}")
print(f"  Final inertia      : {zone_kmeans_model.inertia_:.3f}")
print(f"  Zone size counts:\n{df_maldives['agri_zone_id'].value_counts().sort_index().to_string()}")

## Objective 1 Outputs: Elbow Plot, Zone Map, and Farmer-Oriented Zone Meaning

**AI Purpose:** This cell visualizes the Elbow Method and maps discovered K-Means zones geographically. It then profiles each zone using mean rainfall, NDVI, and latitude to produce interpretable zone narratives.

**Agricultural / Climate Significance:** Each zone can be treated as a management class for farmers. For example, a lower-rainfall northern zone indicates stronger drought risk and supports conservative water-use cover crops, while wetter southern zones can support moisture-demanding biomass cover crops. This transforms unsupervised patterns into practical extension guidance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Objective 1 — Elbow Method and Spatial Agricultural Zones", fontsize=14, fontweight="bold")

# ── Plot 1: Elbow method ─────────────────────────────────────────────────────
axes[0].plot(elbow_k_values, elbow_inertia_values, marker="o", color="#2b8cbe", linewidth=2)
axes[0].axvline(selected_k, linestyle="--", color="#d7301f", label=f"Selected k = {selected_k}")
axes[0].set_title("Elbow Method for K-Means")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Within-Cluster Sum of Squares (Inertia)")
axes[0].legend()

# ── Plot 2: Spatial zone map ──────────────────────────────────────────────────
zone_palette = sns.color_palette("viridis", n_colors=selected_k)
zone_scatter = sns.scatterplot(
    data=df_maldives,
    x="Longitude",
    y="Latitude",
    hue="agri_zone_id",
    palette=zone_palette,
    s=26,
    alpha=0.80,
    edgecolor=None,
    ax=axes[1],
)
axes[1].set_title("K-Means Agricultural Zones Across Maldives Grid")
axes[1].set_xlabel("Longitude (°E)")
axes[1].set_ylabel("Latitude (°N)")
axes[1].legend(title="Zone ID")

plt.tight_layout()

# Save objective-specific visual artifact
_obj1_fig_path = DATA_DIR / "outputs" / "objective1_kmeans_zones.png"
plt.savefig(_obj1_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[Saved] {_obj1_fig_path}")

# ── Zone profiling for interpretation ─────────────────────────────────────────
zone_profile_df: pd.DataFrame = (
    df_maldives.groupby("agri_zone_id", as_index=False)
    .agg(
        sample_count            =("Site_ID", "count"),
        mean_rainfall_mm        =("mean_annual_rainfall_mm", "mean"),
        mean_ndvi_health        =("NDVI_Health", "mean"),
        mean_latitude           =("Latitude", "mean"),
        mean_longitude          =("Longitude", "mean"),
        median_distance_atoll_km=("distance_to_atoll_km", "median"),
    )
    .sort_values("agri_zone_id")
)

print("\n[ Zone Profiles — Objective 1 ]")
print(zone_profile_df.round(3).to_string(index=False))

# Farmer-oriented interpretation helper
def build_farmer_zone_note(mean_rainfall_mm: float, mean_ndvi_health: float, mean_latitude: float) -> str:
    """Generate a concise farmer-facing interpretation for each cluster."""
    if mean_rainfall_mm < 1700:
        rainfall_note = "lower-rainfall zone: prioritize drought-resilient cover crops and tighter irrigation scheduling"
    elif mean_rainfall_mm < 2300:
        rainfall_note = "moderate-rainfall zone: balanced window for mixed cover crop strategies"
    else:
        rainfall_note = "high-rainfall zone: suitable for moisture-demanding biomass cover crops"

    ndvi_note = (
        "vegetation is relatively weak; monitor establishment and nutrient stress closely"
        if mean_ndvi_health < 0
        else "vegetation condition is comparatively stronger"
    )

    latitude_note = (
        "northern-biased zone"
        if mean_latitude > 4.5
        else "southern/central-biased zone"
    )

    return f"{latitude_note}; {rainfall_note}; {ndvi_note}."

print("\n[ What these zones represent for a Maldivian farmer ]")
for _, row in zone_profile_df.iterrows():
    zone_note = build_farmer_zone_note(
        mean_rainfall_mm=row["mean_rainfall_mm"],
        mean_ndvi_health=row["mean_ndvi_health"],
        mean_latitude=row["mean_latitude"],
    )
    print(f"  Zone {int(row['agri_zone_id'])}: {zone_note}")

# Explicit validation print required by project instructions
print("\n[ Validation Check ]")
print(f"  df_maldives shape after clustering: {df_maldives.shape}")
print(f"  NaN in agri_zone_id: {df_maldives['agri_zone_id'].isna().sum()}")
print(df_maldives[["Site_ID", "agri_zone_id", "mean_annual_rainfall_mm", "NDVI_Health", "Latitude"]].head(8).to_string(index=False))

## Objective 2: Feature Selection and Classification for Crop Suitability

**AI Purpose:** This section builds a supervised classification model (Random Forest) to predict `Crop_Type` from current environmental conditions. The model learns non-linear relationships between rainfall, vegetation health, soil chemistry, and local climate to produce class predictions for cover crop suitability.

**Agricultural / Climate Significance:** For the Maldives, this model operationalizes AI-enabled advisory logic by transforming raw environmental data into actionable crop recommendations under monsoon variability, high soil alkalinity (pH 8.0–8.8), and fragmented island geography. The output can support farmers and extension officers in selecting resilient cover crop types under site-specific conditions.

In [ ]:
# Objective 2 dependencies (Colab hint):
# !pip install scikit-learn --quiet

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

# ── Step 1: Define target and feature set ─────────────────────────────────────
# Crop_Type is the supervised label for suitability in this project.
df_maldives["Crop_Type"] = df_maldives["recommended_cover_crop"].astype(str)

classification_feature_columns: list[str] = [
    "mean_annual_rainfall_mm",
    "NDVI_Health",
    "soil_ph",
    "Latitude",
    "air_temperature_c",
    "relative_humidity",
    "soil_n_kg_ha",
    "soil_p_kg_ha",
    "soil_k_kg_ha",
]

classification_target_column: str = "Crop_Type"

# Keep only required columns and handle missing values explicitly
classification_df: pd.DataFrame = df_maldives[
    classification_feature_columns + [classification_target_column]
].copy()

print("[ Objective 2 — Classification Input Validation ]")
print(f"  Shape before NaN handling: {classification_df.shape}")
print(f"  NaN counts:\n{classification_df.isnull().sum().to_string()}")

classification_df = classification_df.dropna().copy()
print(f"  Shape after dropna       : {classification_df.shape}")
print(f"  Class distribution:\n{classification_df[classification_target_column].value_counts().to_string()}")

# ── Step 2: Train-test split ──────────────────────────────────────────────────
X: pd.DataFrame = classification_df[classification_feature_columns]
y: pd.Series = classification_df[classification_target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("\n[ Train/Test Split ]")
print(f"  X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"  y_train classes:\n{y_train.value_counts().to_string()}")
print(f"  y_test classes:\n{y_test.value_counts().to_string()}")

# ── Step 3: Fit Random Forest classifier ──────────────────────────────────────
rf_crop_classifier: RandomForestClassifier = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    n_jobs=-1,
)
rf_crop_classifier.fit(X_train, y_train)

y_pred: np.ndarray = rf_crop_classifier.predict(X_test)

# ── Step 4: Evaluation metrics (assignment requirement) ───────────────────────
objective2_accuracy: float = accuracy_score(y_test, y_pred)
objective2_f1_weighted: float = f1_score(y_test, y_pred, average="weighted")
objective2_confusion: np.ndarray = confusion_matrix(y_test, y_pred, labels=sorted(y.unique()))

print("\n[ Objective 2 — Classification Performance ]")
print(f"  Accuracy Score : {objective2_accuracy:.4f}")
print(f"  F1 Score (w)   : {objective2_f1_weighted:.4f}")
print("\n  Classification Report:")
print(classification_report(y_test, y_pred, digits=4))
print("  Confusion Matrix (raw counts):")
print(pd.DataFrame(
    objective2_confusion,
    index=[f"Actual: {c}" for c in sorted(y.unique())],
    columns=[f"Pred: {c}" for c in sorted(y.unique())],
).to_string())

# ── Step 5: Feature importance analysis ───────────────────────────────────────
rf_feature_importance_df: pd.DataFrame = pd.DataFrame(
    {
        "feature": classification_feature_columns,
        "importance": rf_crop_classifier.feature_importances_,
    }
).sort_values("importance", ascending=False)

print("\n[ Random Forest Feature Importance ]")
print(rf_feature_importance_df.round(4).to_string(index=False))

# Required comparison: Rainfall vs NDVI vs pH
comparison_subset = rf_feature_importance_df[
    rf_feature_importance_df["feature"].isin(["mean_annual_rainfall_mm", "NDVI_Health", "soil_ph"])
].copy()
strongest_predictor_row = comparison_subset.sort_values("importance", ascending=False).iloc[0]

print("\n[ Strongest Predictor Among Rainfall, NDVI, pH ]")
print(comparison_subset.round(4).to_string(index=False))
print(
    f"  Strongest predictor for crop suitability = {strongest_predictor_row['feature']} "
    f"(importance={strongest_predictor_row['importance']:.4f})"
)

# ── Step 6: Visualisations ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle("Objective 2 — Random Forest Crop Suitability Classification", fontsize=14, fontweight="bold")

# Feature importance plot
sns.barplot(
    data=rf_feature_importance_df,
    x="importance",
    y="feature",
    palette="mako",
    ax=axes[0],
)
axes[0].set_title("Feature Importance (Random Forest)")
axes[0].set_xlabel("Importance Score")
axes[0].set_ylabel("Environmental Feature")

# Confusion matrix heatmap
cm_df = pd.DataFrame(
    objective2_confusion,
    index=sorted(y.unique()),
    columns=sorted(y.unique()),
)
sns.heatmap(
    cm_df,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.5,
    ax=axes[1],
)
axes[1].set_title("Confusion Matrix")
axes[1].set_xlabel("Predicted Crop Type")
axes[1].set_ylabel("Actual Crop Type")

plt.tight_layout()

_obj2_fig_path = DATA_DIR / "outputs" / "objective2_rf_classification.png"
plt.savefig(_obj2_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[Saved] {_obj2_fig_path}")

### Objective 2 Interpretation: Why this is a Decision Support Tool

This classifier directly implements the Decision Support logic described in the project context: it converts environmental observations into a recommended cover crop class for operational use. Instead of relying on static, one-size-fits-all guidance, the model links local rainfall conditions, vegetation health status, and alkaline soil context to crop suitability outcomes.

For Maldivian farming systems, this is useful in three ways:
1. **Pre-season planning:** Farmers can choose crop types using expected rainfall and baseline soil conditions before planting.
2. **In-season adjustment:** As NDVI and weather indicators shift, recommendations can be refreshed to reduce climate-shock risk.
3. **Resource targeting:** Extension teams can prioritize advisory support and inputs by identifying where predicted crop suitability is weakest or uncertain.

Therefore, the model functions as an AI-enabled bridge between geospatial monitoring data and practical, field-level cropping decisions across the fragmented atoll landscape.